# 🤖 Agente RAG sobre Sales Dataset — Clase Completa
## Objetivo: construir un bot que busque, filtre y responda preguntas sobre el corpus de ventas

> **El RAG debe ser capaz de:**
> - Ir al corpus (dataset de ventas) y recuperar los registros relevantes
> - Filtrar por producto, país, cliente, estado, año, etc.
> - Decirte **exactamente dónde está** un producto u orden (ciudad, país, dirección)
> - Responder preguntas libres en lenguaje natural, fundamentado en los datos reales

---
### 👨‍🏫 Stack tecnológico
| Componente | Herramienta |
|---|---|
| LLM | **Mistral AI** (mistral-small-latest) |
| Embeddings | sentence-transformers (local, gratis) |
| Vector DB | **FAISS** (local, sin servidor) |
| Framework RAG | **LangChain** |
| Data wrangling | **Pandas** |
| Secrets | **Google Colab Secrets** (forma segura) |


---
## 🗺️ Ruta de Trabajo — Cronograma de la Clase

```
FASE 1 — SETUP (15 min)
  ├── Paso 1 │ Configurar API Key en Colab Secrets   (5 min)
  ├── Paso 2 │ Instalar dependencias                 (5 min)
  └── Paso 3 │ Importar librerías y verificar        (2 min)

FASE 2 — DATA (20 min)
  ├── Paso 4 │ Cargar dataset con Pandas             (3 min)
  ├── Paso 5 │ Exploración EDA con Pandas            (7 min)
  └── Paso 6 │ Normalizar y limpiar el dataset      (10 min)

FASE 3 — RAG PIPELINE (20 min)
  ├── Paso 7 │ Convertir filas → Documentos RAG      (5 min)
  ├── Paso 8 │ Embeddings + índice FAISS             (7 min)
  └── Paso 9 │ Prompt + Cadena RAG con Mistral       (8 min)

FASE 4 — BOT (10 min)
  ├── Paso 10│ Probar con preguntas de ejemplo       (5 min)
  └── Paso 11│ Bot interactivo libre                 (5 min)
```


---
## 📚 Conceptos Clave

### ¿Qué es RAG?
**RAG (Retrieval-Augmented Generation)** combina dos etapas:

```
PREGUNTA DEL USUARIO
        │
        ▼
┌───────────────────┐
│  RETRIEVAL        │  ← El corpus (dataset) se busca semánticamente
│  FAISS + Embeds   │    y se extraen los fragmentos más relevantes
└───────────────────┘
        │
        ▼
┌───────────────────┐
│  GENERATION       │  ← El LLM (Mistral) recibe esos fragmentos
│  Mistral LLM      │    como contexto y genera la respuesta
└───────────────────┘
        │
        ▼
RESPUESTA BASADA EN TUS DATOS REALES
```

### ¿Por qué RAG sobre un dataset tabular?
- El LLM solo conoce hasta su fecha de corte → **no conoce tu dataset**
- RAG le inyecta los datos relevantes en cada pregunta
- El bot puede decirte: *"El producto S10_1678 está en NYC, USA, en 897 Long Airport Avenue"*

### ¿Qué es un Agente?
Un agente puede usar **herramientas** (tools). En esta clase el agente tiene:
- 🔧 `search_orders` — busca órdenes por texto libre
- 🔧 `filter_by_country` — filtra por país
- 🔧 `filter_by_product` — filtra por línea de producto
- 🔧 `find_product_location` — localiza dónde está un producto

---


---
## 🔐 PASO 1 — Configurar API Keys (Forma Segura con Colab Secrets)

### ¿Cómo agregar tu API Key de Mistral?
1. Ir a [https://console.mistral.ai](https://console.mistral.ai) → **API Keys** → crear una
2. En Colab: click en el ícono **🔑 (llave)** del panel izquierdo
3. Click **"+ Add new secret"**
4. Nombre: `MISTRAL_API_KEY` — Valor: tu API Key
5. Activar el toggle **"Notebook access"** ✅

> ⛔ **NUNCA** escribas la API Key directamente en el código.  
> ✅ **SIEMPRE** usá Colab Secrets — el token no queda en el historial ni en el archivo.


In [ ]:
# ─── Cargar API Key desde Colab Secrets ─────────────────────────
from google.colab import userdata
import os

try:
    MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
    os.environ["MISTRAL_API_KEY"] = MISTRAL_API_KEY
    # Mostrar solo los primeros/últimos caracteres para verificar
    masked = MISTRAL_API_KEY[:6] + "..." + MISTRAL_API_KEY[-4:]
    print(f"✅ MISTRAL_API_KEY cargada correctamente: {masked}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("→ Agregá 'MISTRAL_API_KEY' en Colab Secrets (ícono 🔑 del panel izquierdo)")
    raise


---
## 📦 PASO 2 — Instalar Dependencias

### Stack completo:
| Paquete | Para qué sirve |
|---|---|
| `langchain` | Framework principal para RAG y agentes |
| `langchain-mistralai` | Conector oficial Mistral ↔ LangChain |
| `langchain-community` | FAISS, HuggingFace embeddings, tools |
| `faiss-cpu` | Base de datos vectorial local (ultrarrápida) |
| `sentence-transformers` | Modelo de embeddings multilingüe (gratis, local) |
| `pandas` | Carga, limpieza y análisis del dataset |
| `numpy` | Operaciones numéricas |
| `tabulate` | Mostrar tablas en consola |


In [ ]:
# ─── Instalar todo el stack RAG ──────────────────────────────────
print("⏳ Instalando dependencias (puede tardar 2-3 minutos)...")

!pip install -q \
    langchain==0.3.7 \
    langchain-mistralai \
    langchain-community \
    faiss-cpu \
    sentence-transformers \
    pandas \
    numpy \
    tabulate

print("\n✅ Todas las dependencias instaladas correctamente")
print("   langchain, langchain-mistralai, langchain-community")
print("   faiss-cpu, sentence-transformers, pandas, numpy, tabulate")


---
## 🐍 PASO 3 — Importar Librerías y Verificar Versiones


In [ ]:
# ─── SOLUCIÓN AL ERROR DE INCOMPATIBILIDAD ───────────────────────
# Ejecuta esta celda para actualizar numpy y pandas.
# IMPORTANTE: Después de que termine, ve al menú 'Entorno de ejecución'
# (Runtime) -> 'Reiniciar entorno de ejecución' (Restart runtime).

!pip install --upgrade numpy pandas

# Luego de reiniciar, intenta importar nuevamente:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Resto de imports de LangChain
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType

# Utils
from tabulate import tabulate
from IPython.display import display, HTML
import json

print("✅ Librerías importadas correctamente")
print(f"   pandas     : {pd.__version__}")
print(f"   numpy      : {np.__version__}")

---
## 🗃️ PASO 4 — Cargar el Dataset

### Sobre el dataset: `sales_data_sample.csv`
- **2823 órdenes** de ventas globales
- **25 columnas** con info de clientes, productos, ubicaciones y montos
- Rango temporal: 2003–2005
- Países: USA, France, Spain, Australia, UK, Norway y más

### Subir el archivo a Colab:
**Opción A** — Subida manual (ejecutá la celda de carga)  
**Opción B** — Desde Google Drive (montalo primero)  
**Opción C** — Ya lo subiste previamente → el archivo está disponible


In [ ]:
# ─── Opción A: Subida manual desde tu PC ────────────────────────
# Descomentá si necesitás subir el archivo:
from google.colab import files
uploaded = files.upload()  # seleccioná sales_data_sample.csv

# ─── Opción B: Desde Google Drive ───────────────────────────────
# from google.drive import drive
# drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/sales_data_sample.csv'

# ─── Cargar el CSV ───────────────────────────────────────────────
import pandas as pd
CSV_PATH = 'sales_data_normalized.csv'  # ajustá si la ruta es distinta

df_raw = pd.read_csv(CSV_PATH, encoding='latin1')

print(f"✅ Dataset cargado")
print(f"   Filas    : {df_raw.shape[0]:,}")
print(f"   Columnas : {df_raw.shape[1]}")
print(f"\n📋 Primeras 3 filas:")
display(df_raw.head(3))

---
## 🐼 PASO 5 — Exploración con Pandas (EDA)

Antes de construir el RAG necesitamos entender el corpus.  
Usamos Pandas para "chocar" con los datos: tipos, nulos, distribuciones.


In [ ]:
# ═══ EDA COMPLETO ════════════════════════════════════════════════

# ── 1. Tipos de datos y nulos ─────────────────────────────────────
print("=" * 60)
print("📋 TIPOS DE DATOS Y VALORES NULOS")
print("=" * 60)
info_df = pd.DataFrame({
    'Tipo': df_raw.dtypes,
    'No Nulos': df_raw.count(),
    'Nulos': df_raw.isnull().sum(),
    '% Nulos': (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
})
print(tabulate(info_df, headers='keys', tablefmt='grid'))

# ── 2. Estadísticas numéricas ─────────────────────────────────────
print("\n" + "=" * 60)
print("📈 ESTADÍSTICAS — COLUMNAS NUMÉRICAS CLAVE")
print("=" * 60)
num_stats = df_raw[['QUANTITYORDERED','PRICEEACH','SALES','MSRP']].describe().round(2)
display(num_stats)

# ── 3. Valores únicos categóricos ────────────────────────────────
print("\n" + "=" * 60)
print("🏷️  DISTRIBUCIÓN DE VARIABLES CATEGÓRICAS")
print("=" * 60)

for col in ['STATUS', 'PRODUCTLINE', 'DEALSIZE']:
    vc = df_raw[col].value_counts()
    print(f"\n🔹 {col}:")
    for val, cnt in vc.items():
        bar = '█' * int(cnt / 50)
        print(f"   {val:<20} {cnt:>4} {bar}")

# ── 4. Países con más órdenes ─────────────────────────────────────
print("\n" + "=" * 60)
print("🌎 TOP 10 PAÍSES POR NÚMERO DE ÓRDENES")
print("=" * 60)
country_top = df_raw['COUNTRY'].value_counts().head(10)
for country, cnt in country_top.items():
    bar = '█' * int(cnt / 20)
    print(f"   {country:<20} {cnt:>4} {bar}")

# ── 5. Ventas por año ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("📅 VENTAS TOTALES POR AÑO")
print("=" * 60)
yr_sales = df_raw.groupby('YEAR_ID')['SALES'].agg(['sum','count','mean']).round(2)
yr_sales.columns = ['Total $', 'N Órdenes', 'Promedio $']
print(tabulate(yr_sales, headers='keys', tablefmt='grid'))


---
## 🧹 PASO 6 — Normalizar el Dataset con Pandas

La normalización mejora la calidad del corpus que ingresa al RAG:
- ✅ Fechas parseadas
- ✅ Nulos rellenados con valores coherentes
- ✅ Texto limpio (strip + title case)
- ✅ Columnas derivadas para enriquecer el contexto


In [ ]:
# ═══ NORMALIZACIÓN COMPLETA DEL DATASET ═════════════════════════

df = df_raw.copy()

# ── 1. Parsear fechas ─────────────────────────────────────────────
df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'], format='%m/%d/%Y %H:%M',
                                  errors='coerce')
df['ORDER_YEAR']    = df['ORDERDATE'].dt.year.astype('Int64')
df['ORDER_MONTH']   = df['ORDERDATE'].dt.month.astype('Int64')
df['ORDER_DAY']     = df['ORDERDATE'].dt.day.astype('Int64')
df['ORDER_QUARTER'] = df['ORDERDATE'].dt.quarter.astype('Int64')

# ── 2. Rellenar nulos ─────────────────────────────────────────────
df['ADDRESSLINE2'] = df['ADDRESSLINE2'].fillna('').str.strip()
df['STATE']        = df['STATE'].fillna('N/A')
df['POSTALCODE']   = df['POSTALCODE'].fillna('0').astype(str) # Modified: Changed fillna(0).astype(int) to fillna('0').astype(str)
df['TERRITORY']    = df['TERRITORY'].fillna('Sin territorio asignado')

# ── 3. Normalizar texto ───────────────────────────────────────────
str_cols = ['CUSTOMERNAME','CITY','COUNTRY','PRODUCTLINE',
            'STATUS','DEALSIZE','CONTACTLASTNAME','CONTACTFIRSTNAME',
            'ADDRESSLINE1','STATE']
for col in str_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# ── 4. Columnas derivadas ─────────────────────────────────────────
df['REVENUE_CALC']  = (df['QUANTITYORDERED'] * df['PRICEEACH']).round(2)
df['DISCOUNT_PCT']  = ((1 - df['PRICEEACH'] / df['MSRP']) * 100).clip(0).round(1)
df['CONTACT_FULL']  = (df['CONTACTFIRSTNAME'] + ' ' + df['CONTACTLASTNAME']).str.strip()
df['FULL_ADDRESS']  = (
    df['ADDRESSLINE1'].str.strip()
    + df['ADDRESSLINE2'].apply(lambda x: f', {x}' if x else '')
    + ', ' + df['CITY']
    + df['STATE'].apply(lambda x: f', {x}' if x != 'N/A' else '')
    + ', ' + df['COUNTRY']
)

# ── 5. Tipos correctos ────────────────────────────────────────────
df['ORDERNUMBER'] = df['ORDERNUMBER'].astype(str)
df['SALES']       = df['SALES'].round(2)
df['PRICEEACH']   = df['PRICEEACH'].round(2)

# ── 6. Reporte de normalización ───────────────────────────────────
print("✅ NORMALIZACIÓN COMPLETADA")
print(f"   Filas    : {len(df):,}")
print(f"   Columnas : {len(df.columns)} (original: {len(df_raw.columns)} + {len(df.columns)-len(df_raw.columns)} nuevas)")
print(f"   Nulos restantes: {df.isnull().sum().sum()}")
print("\n📊 Columnas nuevas agregadas:")
new_cols = ['ORDER_YEAR','ORDER_MONTH','ORDER_DAY','ORDER_QUARTER',
            'REVENUE_CALC','DISCOUNT_PCT','CONTACT_FULL','FULL_ADDRESS']
for c in new_cols:
    print(f"   ✚ {c}")
print("\n🏠 Ejemplo de FULL_ADDRESS:")
print(df['FULL_ADDRESS'].iloc[0])
print("\n📋 Dataset normalizado (primeras 3 filas):")
display(df[['ORDERNUMBER','ORDERDATE','CUSTOMERNAME','PRODUCTLINE',
            'PRODUCTCODE','FULL_ADDRESS','SALES','STATUS','DEALSIZE']].head(3))

---
## 📄 PASO 7 — Convertir Filas del Dataset → Documentos RAG

Este es el **paso clave** para datos tabulares.  
Cada fila se convierte en un **texto descriptivo rico** que incluye:
- Ubicación exacta (dirección, ciudad, país, territorio)
- Producto (línea, código, MSRP, descuento)
- Transacción (cantidad, precio, total, deal size)
- Estado y fechas

El retriever buscará semánticamente en estos textos para responder  
preguntas como: *"¿Dónde está el producto S10_1678?"*


In [ ]:
# ═══ CONVERTIR CADA FILA EN UN DOCUMENTO TEXTUAL PARA RAG ════════

def row_to_document(row):
    """
    Convierte una fila del DataFrame en un Documento de texto enriquecido.
    La descripción textual es lo que FAISS indexa y el LLM recibe como contexto.
    """
    # Texto principal — narrativo y rico en keywords
    text = (
        f"ORDEN #{row['ORDERNUMBER']} | FECHA: {row['ORDERDATE'].strftime('%d/%m/%Y') if pd.notna(row['ORDERDATE']) else 'N/A'} | "
        f"ESTADO: {row['STATUS']} | DEAL: {row['DEALSIZE']}\n"
        f"CLIENTE: {row['CUSTOMERNAME']} | CONTACTO: {row['CONTACT_FULL']} | TEL: {row['PHONE']}\n"
        f"UBICACIÓN: {row['FULL_ADDRESS']} | CÓDIGO POSTAL: {row['POSTALCODE']} | TERRITORIO: {row['TERRITORY']}\n"
        f"PRODUCTO: {row['PRODUCTLINE']} | CÓDIGO: {row['PRODUCTCODE']} | MSRP: ${row['MSRP']}\n"
        f"PRECIO UNITARIO: ${row['PRICEEACH']} | DESCUENTO: {row['DISCOUNT_PCT']}% | "
        f"CANTIDAD: {row['QUANTITYORDERED']} unidades\n"
        f"VENTA TOTAL: ${row['SALES']} | INGRESOS CALCULADOS: ${row['REVENUE_CALC']}\n"
        f"AÑO: {row['ORDER_YEAR']} | MES: {row['ORDER_MONTH']} | TRIMESTRE: Q{row['ORDER_QUARTER']} | "
        f"LÍNEA DE ORDEN: {row['ORDERLINENUMBER']}"
    )

    # Metadata estructurada — para filtros y referencias exactas
    metadata = {
        "ordernumber"  : str(row['ORDERNUMBER']),
        "customer"     : row['CUSTOMERNAME'],
        "country"      : row['COUNTRY'],
        "city"         : row['CITY'],
        "territory"    : row['TERRITORY'],
        "address"      : row['FULL_ADDRESS'],
        "productline"  : row['PRODUCTLINE'],
        "productcode"  : row['PRODUCTCODE'],
        "status"       : row['STATUS'],
        "dealsize"     : row['DEALSIZE'],
        "year"         : int(row['ORDER_YEAR']) if pd.notna(row['ORDER_YEAR']) else 0,
        "sales"        : float(row['SALES']),
        "discount_pct" : float(row['DISCOUNT_PCT']),
        "quantity"     : int(row['QUANTITYORDERED']),
    }
    return Document(page_content=text, metadata=metadata)

# Convertir todas las filas
print("⏳ Convirtiendo 2823 filas a documentos RAG...")
documents = [row_to_document(row) for _, row in df.iterrows()]

print(f"\n✅ {len(documents):,} documentos creados")
print("\n" + "─"*70)
print("📄 EJEMPLO DE DOCUMENTO GENERADO:")
print("─"*70)
print(documents[0].page_content)
print("─"*70)
print("\n🏷️  METADATA DEL DOCUMENTO:")
for k, v in documents[0].metadata.items():
    print(f"   {k:<14}: {v}")


---
## 🔍 PASO 8 — Embeddings + Índice Vectorial FAISS

### ¿Qué son los embeddings?
Son representaciones numéricas de texto (vectores de ~384 dimensiones).  
Textos semánticamente similares tienen vectores cercanos en el espacio.

### Pipeline:
```
"¿Dónde está el producto S10_1678?"
              │
              ▼  [Embedding model]
    [0.12, -0.34, 0.87, ...]   ← vector de la pregunta
              │
              ▼  [FAISS búsqueda de vecinos más cercanos]
    Documentos del corpus con vectores similares
    → Órdenes de S10_1678 con sus direcciones completas
```


In [ ]:
# ═══ EMBEDDINGS + CONSTRUCCIÓN DEL ÍNDICE FAISS ═════════════════

print("⏳ Cargando modelo de embeddings multilingüe...")
print("   Modelo: paraphrase-multilingual-MiniLM-L12-v2")
print("   (soporta español, inglés y otros — ideal para este dataset)\n")

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Modelo de embeddings cargado (local, sin API Key)")
print("\n⏳ Construyendo índice FAISS sobre 2823 documentos...")
print("   (puede tardar 1-2 minutos)\n")

# Construir el vectorstore
vectorstore = FAISS.from_documents(documents, embeddings_model)

# Guardar localmente
vectorstore.save_local("faiss_sales_index")

print(f"✅ Índice FAISS construido y guardado")
print(f"   Vectores indexados : {vectorstore.index.ntotal:,}")
print(f"   Dimensión          : {vectorstore.index.d}")
print(f"   Guardado en        : faiss_sales_index/")

# ── Verificar con búsquedas de prueba ────────────────────────────
print("\n🔎 TESTS DE BÚSQUEDA SEMÁNTICA:")
tests = [
    ("producto S10_1678 en USA", 2),
    ("órdenes canceladas classic cars", 2),
    ("cliente en Francia motorcycles", 2),
]
for query, k in tests:
    results = vectorstore.similarity_search(query, k=k)
    print(f"\n  Query: '{query}'")
    for i, doc in enumerate(results, 1):
        m = doc.metadata
        print(f"    [{i}] Orden #{m['ordernumber']} | {m['customer']} | "
              f"{m['productline']} | {m['city']}, {m['country']}")


---
## 🦾 PASO 9 — Configurar el Agente RAG con Mistral

### Arquitectura del Agente
El agente tiene **herramientas especializadas** que puede invocar:

| Tool | ¿Qué hace? |
|---|---|
| `buscar_en_corpus` | Búsqueda semántica libre en todo el dataset |
| `localizar_producto` | Dice exactamente dónde está un producto (ciudad, país, dirección) |
| `filtrar_por_pais` | Filtra órdenes por país |
| `filtrar_por_producto` | Filtra por línea de producto |
| `resumen_estadistico` | Estadísticas numéricas del dataset |

El LLM (Mistral) decide qué herramienta usar según la pregunta.


In [ ]:
# ═══ CONFIGURAR MISTRAL LLM ══════════════════════════════════════
from google.colab import userdata
import os

try:
    MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
    os.environ["MISTRAL_API_KEY"] = MISTRAL_API_KEY
    # Display only masked key for security
    masked = MISTRAL_API_KEY[:6] + "..." + MISTRAL_API_KEY[-4:]
    print(f"✅ MISTRAL_API_KEY cargada correctamente: {masked}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("→ Agregá 'MISTRAL_API_KEY' en Colab Secrets (ícono 🔑 del panel izquierdo)")
    raise

llm = ChatMistralAI(
    model="mistral-small-latest",
    mistral_api_key=MISTRAL_API_KEY,
    temperature=0.05,   # muy bajo → respuestas factuales y precisas
    max_tokens=1500
)
print("✅ Mistral LLM configurado (mistral-small-latest)")

# ═══ RETRIEVER BASE ═══════════════════════════════════════════════
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

# ═══ PROMPT PERSONALIZADO ════════════════════════════════════════
SYSTEM_PROMPT = """Sos un asistente experto en análisis del corpus de ventas.
Tu trabajo es responder preguntas usando ÚNICAMENTE la información del contexto proporcionado.

REGLAS ESTRICTAS:
1. Basate SOLO en los datos del contexto. No inventes ni asumas nada.
2. Si la pregunta es sobre UBICACIÓN, indicá siempre: dirección, ciudad, país y territorio.
3. Si la pregunta es sobre un PRODUCTO, indicá: código, línea, MSRP, descuento y dónde se vendió.
4. Si hay MÚLTIPLES resultados, listalos en formato claro.
5. Si no encontrás la información, decí: "No encontré esa información en el corpus."
6. Citá el número de orden cuando sea relevante.
7. Respondé en español, de forma concisa y estructurada.

CONTEXTO DEL CORPUS:
{context}

PREGUNTA: {question}

RESPUESTA ESTRUCTURADA:"""

rag_prompt = PromptTemplate(
    template=SYSTEM_PROMPT,
    input_variables=["context", "question"]
)

# ═══ CADENA RAG BASE ══════════════════════════════════════════════
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": rag_prompt},
    return_source_documents=True
)
print("✅ Cadena RAG base configurada")

# ═══ TOOLS ESPECIALIZADAS DEL AGENTE ═════════════════════════════

def buscar_en_corpus(query: str) -> str:
    """Búsqueda semántica libre en el corpus de ventas."""
    result = qa_chain.invoke({"query": query})
    return result["result"]

def localizar_producto(producto: str) -> str:
    """Localiza exactamente dónde está/fue vendido un producto."""
    # Filtrar documentos con ese productcode o productline
    query_text = f"ubicación dirección ciudad país de {producto}"
    docs = vectorstore.similarity_search(query_text, k=10)

    ubicaciones = []
    vistos = set()
    for doc in docs:
        m = doc.metadata
        key = (m['productcode'], m['customer'])
        if key not in vistos:
            vistos.add(key)
            if (producto.upper() in m['productcode'].upper() or
                producto.lower() in m['productline'].lower() or
                producto.lower() in m['customer'].lower()):
                ubicaciones.append({
                    'Orden': m['ordernumber'],
                    'Producto': m['productcode'],
                    'Línea': m['productline'],
                    'Cliente': m['customer'],
                    'Dirección': m['address'],
                    'País': m['country'],
                    'Territorio': m['territory'],
                    'Venta $': m['sales'],
                    'Estado': m['status'],
                })

    if not ubicaciones:
        # Fallback al RAG semántico
        result = qa_chain.invoke({"query": f"¿Dónde está el producto {producto}? Dame dirección, ciudad y país."})
        return result["result"]

    respuesta = f"📍 UBICACIONES DE '{producto.upper()}':\n"
    respuesta += "─" * 60 + "\n"
    for u in ubicaciones[:8]:
        respuesta += (
            f"  Orden #{u['Orden']} | {u['Producto']} ({u['Línea']})\n"
            f"  Cliente : {u['Cliente']}\n"
            f"  Dirección: {u['Dirección']}\n"
            f"  País    : {u['País']} | Territorio: {u['Territorio']}\n"
            f"  Venta   : ${u['Venta $']} | Estado: {u['Estado']}\n"
            f"  {'─'*55}\n"
        )
    return respuesta

def filtrar_por_pais(pais: str) -> str:
    """Filtra y resume órdenes de un país específico."""
    pais_norm = pais.strip().title()
    subset = df[df['COUNTRY'].str.contains(pais_norm, case=False, na=False)]

    if subset.empty:
        return f"No se encontraron órdenes para el país: {pais}"

    resumen = (
        f"📊 ÓRDENES DE {pais_norm.upper()}:\n"
        f"  Total órdenes    : {len(subset):,}\n"
        f"  Ventas totales   : ${subset['SALES'].sum():,.2f}\n"
        f"  Venta promedio   : ${subset['SALES'].mean():,.2f}\n"
        f"  Clientes únicos  : {subset['CUSTOMERNAME'].nunique()}\n"
        f"  Productos        : {', '.join(subset['PRODUCTLINE'].unique())}\n"
        f"  Estados          : {', '.join(subset['STATUS'].unique())}\n"
        f"\n  Top clientes:\n"
    )
    top = subset.groupby('CUSTOMERNAME')['SALES'].sum().nlargest(5)
    for cust, sales in top.items():
        resumen += f"    • {cust}: ${sales:,.2f}\n"
    return resumen

def filtrar_por_producto(linea: str) -> str:
    """Filtra y resume órdenes por línea de producto."""
    linea_norm = linea.strip().title()
    subset = df[df['PRODUCTLINE'].str.contains(linea_norm, case=False, na=False)]

    if subset.empty:
        return f"No se encontraron órdenes para la línea: {linea}"

    resumen = (
        f"🚗 LÍNEA DE PRODUCTO: {linea_norm.upper()}\n"
        f"  Total órdenes    : {len(subset):,}\n"
        f"  Ventas totales   : ${subset['SALES'].sum():,.2f}\n"
        f"  Precio promedio  : ${subset['PRICEEACH'].mean():,.2f}\n"
        f"  Descuento prom.  : {subset['DISCOUNT_PCT'].mean():.1f}%\n"
        f"  MSRP promedio    : ${subset['MSRP'].mean():,.2f}\n"
        f"  Países activos   : {', '.join(subset['COUNTRY'].unique()[:8])}\n"
        f"  Deal sizes       : {subset['DEALSIZE'].value_counts().to_dict()}\n"
        f"\n  Productos más vendidos (códigos):\n"
    )
    top_prod = subset.groupby('PRODUCTCODE')['SALES'].sum().nlargest(5)
    for code, sales in top_prod.items():
        resumen += f"    • {code}: ${sales:,.2f}\n"
    return resumen

def resumen_estadistico(parametro: str) -> str:
    """Devuelve estadísticas generales o por parámetro del dataset."""
    total_ventas = df['SALES'].sum()
    resumen = (
        f"📈 RESUMEN ESTADÍSTICO DEL DATASET\n"
        f"{'─'*50}\n"
        f"  Total filas           : {len(df):,}\n"
        f"  Ventas totales        : ${total_ventas:,.2f}\n"
        f"  Venta máxima          : ${df['SALES'].max():,.2f}\n"
        f"  Venta mínima          : ${df['SALES'].min():,.2f}\n"
        f"  Venta promedio        : ${df['SALES'].mean():,.2f}\n"
        f"  Años en el dataset    : {sorted(df['ORDER_YEAR'].dropna().unique().astype(int).tolist())}\n"
        f"  Países               : {df['COUNTRY'].nunique()} únicos\n"
        f"  Clientes             : {df['CUSTOMERNAME'].nunique()} únicos\n"
        f"  Líneas de producto   : {', '.join(df['PRODUCTLINE'].unique())}\n"
        f"  Deal sizes           : {df['DEALSIZE'].value_counts().to_dict()}\n"
        f"  Estados de órdenes   : {df['STATUS'].value_counts().to_dict()}\n"
    )
    return resumen

# ═══ DEFINIR TOOLS PARA EL AGENTE ════════════════════════════════
tools = [
    Tool(
        name="buscar_en_corpus",
        func=buscar_en_corpus,
        description=(
            "Busca información en el corpus de ventas con lenguaje natural. "
            "Úsala para preguntas generales sobre órdenes, clientes, fechas o ventas."
        )
    ),
    Tool(
        name="localizar_producto",
        func=localizar_producto,
        description=(
            "Localiza dónde está un producto en el corpus: devuelve dirección completa, "
            "ciudad, país y territorio. Input: código de producto o nombre de línea."
        )
    ),
    Tool(
        name="filtrar_por_pais",
        func=filtrar_por_pais,
        description=(
            "Filtra y resume todas las órdenes de un país específico. "
            "Input: nombre del país en inglés o español."
        )
    ),
    Tool(
        name="filtrar_por_producto",
        func=filtrar_por_producto,
        description=(
            "Filtra y resume ventas por línea de producto: Classic Cars, Motorcycles, "
            "Vintage Cars, Trucks and Buses, Planes, Ships, Trains."
        )
    ),
    Tool(
        name="resumen_estadistico",
        func=resumen_estadistico,
        description=(
            "Devuelve estadísticas generales del dataset completo: totales, promedios, "
            "distribuciones. Input: cualquier texto (ej: 'general')."
        )
    ),
]

# ═══ INICIALIZAR EL AGENTE ════════════════════════════════════════
agente = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=4,
    handle_parsing_errors=True
)

print("\n✅ AGENTE RAG configurado con 5 herramientas especializadas")
print("   • buscar_en_corpus     — búsqueda semántica libre")
print("   • localizar_producto   — ubicación exacta de productos")
print("   • filtrar_por_pais     — análisis por país")
print("   • filtrar_por_producto — análisis por línea de producto")
print("   • resumen_estadistico  — estadísticas generales")

---
## 🧪 PASO 10 — Probar el Agente con Preguntas de Ejemplo

Probamos el agente con los tipos de preguntas que debe responder:
1. **Localización de producto** — ¿dónde está X?
2. **Filtro por país** — clientes/ventas en Y
3. **Estado de órdenes** — canceladas, en proceso, etc.
4. **Estadísticas** — totales, promedios, comparaciones
5. **Pregunta compleja** — combinando varios filtros


In [ ]:
# ─── Función helper para mostrar respuestas ──────────────────────
def preguntar(pregunta: str, usar_agente: bool = True):
    print("\n" + "╔" + "═"*66 + "╗")
    print(f"║  ❓ {pregunta[:62]:<62} ║")
    print("╚" + "═"*66 + "╝")

    if usar_agente:
        try:
            respuesta = agente.run(pregunta)
        except Exception as e:
            respuesta = buscar_en_corpus(pregunta)
    else:
        resultado = qa_chain.invoke({"query": pregunta})
        respuesta = resultado["result"]

    print("\n🤖 RESPUESTA DEL AGENTE RAG:")
    print("─" * 68)
    print(respuesta)
    print("─" * 68)
    return respuesta


In [ ]:
# ──────────────────────────────────────────────────────────────
# TEST 1: Localización de producto
# ──────────────────────────────────────────────────────────────
preguntar("¿Dónde está el producto S10_1678? Dame la dirección exacta, ciudad y país.")


In [ ]:
# ──────────────────────────────────────────────────────────────
# TEST 2: Filtro por país
# ──────────────────────────────────────────────────────────────
preguntar("¿Qué órdenes hay de clientes en Francia? ¿Cuánto es el total de ventas?")


In [ ]:
# ──────────────────────────────────────────────────────────────
# TEST 3: Estado de órdenes
# ──────────────────────────────────────────────────────────────
preguntar("¿Hay órdenes canceladas? ¿De qué productos son y en qué países están?")


In [ ]:
# ──────────────────────────────────────────────────────────────
# TEST 4: Línea de producto + ubicación
# ──────────────────────────────────────────────────────────────
preguntar("¿En qué países se venden motocicletas? ¿Cuál es la orden de mayor venta?")


In [ ]:
# ──────────────────────────────────────────────────────────────
# TEST 5: Pregunta compleja combinada
# ──────────────────────────────────────────────────────────────
preguntar("Dame un resumen estadístico completo del dataset de ventas.")


In [ ]:
# ──────────────────────────────────────────────────────────────
# TEST 6: Pregunta directa sobre ubicación
# ──────────────────────────────────────────────────────────────
preguntar("¿En qué ciudad y dirección está el cliente Land Of Toys?")


---
## 💬 PASO 11 — Bot Interactivo — Tu turno de preguntar

Ejecutá la siguiente celda para iniciar el chat interactivo.  
El agente decidirá automáticamente qué herramienta usar para cada pregunta.

### Ejemplos de preguntas que podés hacer:
- `¿Dónde está el producto S18_4409?`
- `¿Qué clientes hay en Australia?`
- `¿Cuál es el deal más grande del 2004?`
- `¿Hay órdenes en proceso? ¿Cuáles son y dónde están?`
- `¿Qué línea de producto tiene más ventas?`
- `Dame los Classic Cars vendidos en España`

Escribí `salir` para terminar.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 🤖 BOT RAG INTERACTIVO — SALES DATASET
# ══════════════════════════════════════════════════════════════
print("╔══════════════════════════════════════════════════════════╗")
print("║   🤖 RAG BOT — Sales Dataset │ Powered by Mistral AI   ║")
print("║   📊 Corpus: 2823 órdenes │ 7 líneas │ 19 países        ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Herramientas disponibles:                               ║")
print("║   🔍 Búsqueda semántica en todo el corpus               ║")
print("║   📍 Localización exacta de productos                   ║")
print("║   🌎 Filtro por país                                    ║")
print("║   🚗 Filtro por línea de producto                       ║")
print("║   📈 Estadísticas y resúmenes                           ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Escribí 'salir' para terminar │ 'ayuda' para ejemplos  ║")
print("╚══════════════════════════════════════════════════════════╝")

EJEMPLOS = [
    "¿Dónde está el producto S10_1678?",
    "¿Qué clientes hay en Australia?",
    "¿Cuál es el deal más grande del 2004?",
    "Mostrame órdenes en proceso",
    "Dame un resumen estadístico",
    "¿Qué Classic Cars se vendieron en España?",
]

while True:
    print()
    user_input = input("💬 Tu pregunta: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ['salir', 'exit', 'quit', 'q']:
        print("\n👋 ¡Hasta luego! Bot finalizado.")
        break

    if user_input.lower() == 'ayuda':
        print("\n💡 Ejemplos de preguntas:")
        for i, ej in enumerate(EJEMPLOS, 1):
            print(f"   {i}. {ej}")
        continue

    print("\n⏳ Procesando con el agente RAG...")
    try:
        respuesta = agente.run(user_input)
        print("\n🤖 RESPUESTA:")
        print("─" * 65)
        print(respuesta)
        print("─" * 65)
    except Exception as e:
        # Fallback a la cadena RAG directa
        try:
            resultado = qa_chain.invoke({"query": user_input})
            print("\n🤖 RESPUESTA (RAG directo):")
            print("─" * 65)
            print(resultado["result"])
            print("─" * 65)
        except Exception as e2:
            print(f"❌ Error: {e2}")


---
## 📊 Apéndice — Análisis Pandas Complementario

Consultas directas al DataFrame para validar respuestas del bot o hacer análisis más profundos.


In [ ]:
# ─── Top 10 órdenes por monto de venta ───────────────────────────
print("🏆 TOP 10 ÓRDENES POR MONTO DE VENTA")
top10 = df.nlargest(10, 'SALES')[
    ['ORDERNUMBER','CUSTOMERNAME','PRODUCTLINE','PRODUCTCODE',
     'SALES','STATUS','CITY','COUNTRY','FULL_ADDRESS']
].reset_index(drop=True)
display(top10)


In [ ]:
# ─── Ventas por país y línea de producto (pivot) ──────────────────
print("📊 VENTAS POR PAÍS Y LÍNEA DE PRODUCTO")
pivot = df.pivot_table(
    values='SALES',
    index='COUNTRY',
    columns='PRODUCTLINE',
    aggfunc='sum',
    fill_value=0
).round(0).astype(int)
pivot['TOTAL'] = pivot.sum(axis=1)
pivot = pivot.sort_values('TOTAL', ascending=False)
display(pivot.head(15))


In [ ]:
# ─── Productos únicos con su ubicación de venta ──────────────────
print("📍 CADA CÓDIGO DE PRODUCTO → CIUDADES DONDE FUE VENDIDO")
prod_loc = (
    df.groupby(['PRODUCTCODE', 'PRODUCTLINE'])
    .apply(lambda x: ', '.join(sorted(x['CITY'].unique()[:5])))
    .reset_index()
    .rename(columns={0: 'Ciudades'})
)
display(prod_loc.head(20))


In [ ]:
# ─── Órdenes por estado con totales ──────────────────────────────
print("📋 ÓRDENES POR ESTADO")
status_summary = df.groupby('STATUS').agg(
    N_Ordenes=('ORDERNUMBER','count'),
    Venta_Total=('SALES','sum'),
    Venta_Promedio=('SALES','mean'),
    Paises=('COUNTRY', lambda x: x.nunique()),
).round(2)
display(status_summary)
